# Accelerated Data Science with RAPIDS #

## 04 - cuGraph as a NetworkX backend  ##

**Table of Contents**
<br>
This notebook introduces the various methods of utilizing the cuGraph backend for NetworkX and runs centrality algorithms on the dataset. This notebook covers the below sections:
1. [Background](#Background)
2. [Installation](#Installation)
3. [Utilizing nx-cugraph](#Utilizing-nx-cugraph)
    * [Runtime Environment Variable](#Runtime-Environment-Variable)
    * [Backend Keyword Argument](#Backend-Keyword-Argument)
    * [Type-Based Dispatching](#Type-Based-Dispatching)
4. [Computing Centrality](#Computing-Centrality)
    * [Creating Graph](#Creating-Graph)
    * [Running Centrality Algorithms](#Running-Centrality-Algorithms)
    * [Betweenness Centrality](#Betweenness-Centrality)
    * [Degree Centrality](#Degree-Centrality)
    * [Katz Centrality](#Katz-Centrality)
    * [Pagerank Centrality](#Pagerank-Centrality)
    * [Eigenvector Centrality](#Eigenvector-Centrality)
    * [Visualize Results](#Visualize-Results)
    * [Exercise #1 - Type Dispatch](#Exercise-#1---Type-Dispatch)

## Background ##
RAPIDS recently introduced a new backend to NetworkX called nx-cugraph. With this backend, you can automatically accelerate supported algorithms. In this notebook, we will cover the various methods of enabling the cugraph backend, and use the backend to run different centrality algorithms.

## Installation ##
We have already prepared the environment with nx-cugraph installed. When you are using your own environment, below is the command for installation. 

## Utilizing nx-cugraph ##
There are 3 ways to utilize nx-cugraph

1. **Environment Variable at Runtime**
2. **Backend keyword argument**
3. **Type-Based dispatching**

Let's dig a little deeper in to each of these methods.

### Runtime Environment Variable ###
The NETWORKX_AUTOMATIC_BACKENDS environment variable can be used to have NetworkX automatically dispatch to specified backends. Set NETWORKX_AUTOMATIC_BACKENDS=cugraph to use nx-cugraph to GPU accelerate supported APIs with no code changes. We will also be loading the cuDF pandas module to accelerate csv loading.

In [ ]:
!NETWORKX_AUTOMATIC_BACKENDS=cugraph python -m cudf.pandas scripts/networkx.py

### Backend Keyword Argument ###
NetworkX also supports explicitly specifying a particular backend for supported APIs with the backend= keyword argument. This argument takes precedence over the NETWORKX_AUTOMATIC_BACKENDS environment variable. This method also requires that the specified backend already be installed.

In [2]:
import warnings
warnings.filterwarnings('ignore')

%load_ext cudf.pandas
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# This cell must run to import the data for the example

import os

dir_path = '/content/drive/MyDrive/Accel_DS_RAPIDS'
file_path_nodes = 'part2/data/road_nodes.csv'
file_path_road = 'part2/data/road_graph.csv'
file_path_speed = 'part2/data/road_graph_speed.csv'

print(f"Checking existence of directory: {dir_path}")
if os.path.exists(dir_path) and os.path.isdir(dir_path):
    print(f"The directory '{dir_path}' exists.")
else:
    print(f"Creating '{dir_path}'.")
    # Create data folder if it doesn't exist
    os.makedirs(dir_path, exist_ok=True)
absolute_path_to_file_nodes = os.path.join(dir_path, file_path_nodes)
absolute_path_to_file_road = os.path.join(dir_path, file_path_road)
absolute_path_to_file_speed = os.path.join(dir_path, file_path_speed)

url_dict = {
    file_path_nodes: "https://drive.google.com/uc?id=14cfIMfR4YXBIkZazGDuj-O6k4T7YBu72", 
    file_path_road: "https://drive.google.com/uc?id=1fOApjOcfTm-y25mtCCCw8iUKZTuT-fg-",
    file_path_speed: "https://drive.google.com/uc?id=1M8cA1Ix1XxPs1346ZYffAKRxMMXaZjBX", 
}

for file_path, absolute_path in [(file_path_nodes, absolute_path_to_file_nodes), (file_path_road, absolute_path_to_file_road), (file_path_speed, absolute_path_to_file_speed)]:
    print(f"\nChecking existence of file: {absolute_path}")
    if os.path.exists(absolute_path):
        print(f"The file '{file_path}' exists.")
    else:
        print(f"The file '{file_path}' not found.")
        print(f"Downloading ")
        # Download the file from Google Drive
        import gdown
        url = url_dict.get(file_path)
        gdown.download(url, absolute_path)



In [ ]:

# Load the CSV file
road_graph = pd.read_csv(absolute_path_to_file_road, dtype=['int32', 'int32', 'float32'], nrows=1000)


In [4]:

# Create an empty graph
G = nx.from_pandas_edgelist(road_graph, source='src', target='dst', edge_attr='length')
b = nx.betweenness_centrality(G, k=1000, backend="cugraph")

### Type-Based Dispatching ###
For users wanting to ensure a particular behavior, without the potential for runtime conversions, NetworkX offers type-based dispatching. To utilize this method, users must import the desired backend and create a Graph instance for it.

In [5]:
import networkx as nx
import nx_cugraph as nxcg

# Loading data from previous cell
G = nx.from_pandas_edgelist(road_graph, source='src', target='dst', edge_attr='length') 

nxcg_G = nxcg.from_networkx(G)             # conversion happens once here
b = nx.betweenness_centrality(nxcg_G, k=1000)  # nxcg Graph type causes cugraph backend to be used, no conversion necessary

## Computing Centrality ##
Now that we learned how to enable nx-cugraph, let's try to use it in a workflow! We will be using the backend argument for this example. First let's create a graph.

### Creating Graph ###

In [6]:
# Create a graph from already loaded dataframe
G = nx.from_pandas_edgelist(road_graph, source='src', target='dst', edge_attr='length')

### Running Centrality Algorithms ###
Now, let's run the various centrality algorithms!

### Betweenness Centrality ###
**Definition:** Measures how often a node lies on the shortest path between other nodes in the network.

**Formula:** For each node, count how many shortest paths pass through it, normalized by total possible paths.

**Why It Matters:**
- **Identifies bottlenecks** - Nodes with high betweenness control information flow
- **Critical in road networks** - Major intersections that connect different areas
- **Failure impact** - Removing high-betweenness nodes fragments the network
- **Real example:** A bridge connecting two cities has high betweenness

**Interpretation:** High values = critical connector/gatekeeper in the network

In [7]:
b = nx.betweenness_centrality(G, backend="cugraph")

### Degree Centrality ###
**Definition:** The simplest measure - counts the number of direct connections a node has.

**Formula:** Number of edges connected to a node / Maximum possible connections

**Why It Matters:**
- **Easiest to compute** - Just count connections
- **Immediate influence** - More connections = more direct reach
- **Local importance** - Identifies hubs in the network
- **Real example:** A major airport with flights to many cities

**Interpretation:** High values = highly connected nodes with broad local reach

**Limitation:** Doesn't consider the importance of neighbors - all connections weighted equally

In [8]:
d = nx.degree_centrality(G, backend="cugraph")

### Katz Centrality ###
**Definition:** Measures influence by counting ALL paths from a node, not just shortest paths. Nearby connections matter more than distant ones.

**Formula:** Sum of weighted paths of all lengths, where longer paths have exponentially decreasing weights (controlled by alpha parameter)

**Why It Matters:**
- **Global influence** - Considers the entire network structure
- **Weighted by distance** - Direct connections matter most, but indirect ones still count
- **Better than degree** - Being connected to important nodes increases your score
- **Real example:** A person knowing influential people has higher Katz centrality

**Interpretation:** High values = globally well-positioned with strong direct and indirect reach

**Vs PageRank:** Katz uses attenuation for distance; PageRank uses probability of random walks

In [9]:
k = nx.katz_centrality(G, backend="cugraph")

### Pagerank Centrality ###
**Definition:** Google's original algorithm - measures importance based on the quality and quantity of incoming links. A node is important if important nodes link to it.

**Formula:** Probability that a random walker ends up at a node (with occasional random jumps)

**Why It Matters:**
- **Quality over quantity** - Links from important nodes count more
- **Recursive definition** - Importance is self-reinforcing
- **Web ranking** - Originally designed to rank web pages
- **Real example:** Academic papers cited by highly-cited papers have high PageRank

**Interpretation:** High values = prestigious nodes that other important nodes reference

**Key Insight:** Not just about having many connections, but about WHO connects to you

In [10]:
p = nx.pagerank(G, max_iter=10, tol=1.0e-3, backend="cugraph")

### Eigenvector Centrality ###
**Definition:** A node's importance is proportional to the sum of the importance of its neighbors. Similar to PageRank but considers all connections (not just incoming).

**Formula:** Based on the principal eigenvector of the adjacency matrix - each node's score is the weighted sum of its neighbors' scores

**Why It Matters:**
- **Network prestige** - Connected to important nodes = you're important
- **Recursive quality** - Mutual reinforcement of importance
- **Social influence** - Having influential friends makes you influential
- **Real example:** Being friends with celebrities increases your eigenvector centrality

**Interpretation:** High values = connected to other well-connected, important nodes

**Vs PageRank:** Eigenvector treats all edges equally; PageRank focuses on directed incoming links and adds random jumps for stability

**Limitation:** Can give zero scores to nodes in disconnected components or periphery

In [11]:
e = nx.eigenvector_centrality(G, max_iter=1000, tol=1.0e-3, backend="cugraph")

### Visualize Results ###
Now let's visualize results! We will only display the top 5 rows for readibility. 

In [12]:
from IPython.display import display_html
dc_top = pd.DataFrame(sorted(d.items(), key=lambda x:x[1], reverse=True)[:5], columns=["vertex", "degree_centrality"])
bc_top = pd.DataFrame(sorted(b.items(), key=lambda x:x[1], reverse=True)[:5], columns=["vertex", "betweenness_centrality"])
katz_top = pd.DataFrame(sorted(k.items(), key=lambda x:x[1], reverse=True)[:5], columns=["vertex", "katz_centrality"])
pr_top = pd.DataFrame(sorted(p.items(), key=lambda x:x[1], reverse=True)[:5], columns=["vertex", "pagerank"])
ev_top = pd.DataFrame(sorted(e.items(), key=lambda x:x[1], reverse=True)[:5], columns=["vertex", "eigenvector_centrality"])

df1_styler = dc_top.style.set_table_attributes("style='display:inline'").set_caption('Degree').hide(axis='index')
df2_styler = bc_top.style.set_table_attributes("style='display:inline'").set_caption('Betweenness').hide(axis='index')
df3_styler = katz_top.style.set_table_attributes("style='display:inline'").set_caption('Katz').hide(axis='index')
df4_styler = pr_top.style.set_table_attributes("style='display:inline'").set_caption('PageRank').hide(axis='index')
df5_styler = ev_top.style.set_table_attributes("style='display:inline'").set_caption('EigenVector').hide(axis='index')

display_html(df1_styler._repr_html_()+df2_styler._repr_html_()+df3_styler._repr_html_()+df4_styler._repr_html_()+df5_styler._repr_html_(), raw=True)

vertex,degree_centrality
24,0.002847
72,0.002847
86,0.002847
127,0.002847
133,0.002847
vertex,betweenness_centrality
222,0.000007
381,0.000007
24,0.000006
72,0.000006


### Exercise #1 - Type Dispatch ###
Use the type dispatching method to obtain pagerank centrality results with the cugraph backend.

In [ ]:

import networkx as nx
import nx_cugraph as nxcg

# Loading data from previous cell
G = nx.from_pandas_edgelist(road_graph, source='src', target='dst', edge_attr='length') 

nxcg_G = <<<<FIXME>>>>             # conversion happens once here
p = <<<<FIXME>>>>(nxcg_G, max_iter=10, tol=1.0e-3) # nxcg Graph type causes cugraph backend to be used, no conversion necessary

pd.DataFrame(sorted(p.items(), key=lambda x:x[1], reverse=True)[:5], columns=["vertex", "pagerank"])

,vertex,pagerank
0,24,0.002525
1,72,0.002525
2,86,0.002525
3,127,0.002525
4,133,0.002525


<details style='border:1px solid #d0d7de; border-radius:10px; padding:0.45em 0.8em; background:linear-gradient(180deg,#f8fbff 0%,#ffffff 100%); box-shadow:0 1px 3px rgba(0,0,0,0.06);'>
<summary style='font-weight:600; color:#0b57d0; cursor:pointer;'>Show Solution</summary>

```python
import networkx as nx
import nx_cugraph as nxcg

# Loading data from previous cell
G = nx.from_pandas_edgelist(road_graph, source='src', target='dst', edge_attr='length') 

nxcg_G = nxcg.from_networkx(G)             # conversion happens once here
p = nx.pagerank(nxcg_G, max_iter=10, tol=1.0e-3) # nxcg Graph type causes cugraph backend to be used, no conversion necessary

pd.DataFrame(sorted(p.items(), key=lambda x:x[1], reverse=True)[:5], columns=["vertex", "pagerank"])
```
</details>